# KVO_08 — Test Musanga Earth Engine Cloud project

**Purpose:** diagnose why `musanga-511110` does not appear in the *Earth Engine enabled Cloud Projects* list. This notebook **does not upload imagery, create assets, or modify the project**.

Use a fresh Colab session and select **vanoost@gmail.com** in the Earth Engine authentication window (avoid the UCLouvain account for this test).

The tests distinguish (1) authentication, (2) Earth Engine initialization, (3) actual server-side computation, and (4) asset listing permissions. The first three succeeding establishes practical Earth Engine access even if the Code Editor's project list has not updated. The asset-listing test may require separate IAM rights.

**Project:** `musanga-511110` · **Expected account:** `vanoost@gmail.com`


In [ ]:
%pip -q install -U earthengine-api google-auth

In [ ]:
# BLOCK 1 — Explicit Earth Engine authentication and project initialization
import ee, traceback
PROJECT = "musanga-511110"
EXPECTED_ACCOUNT = "vanoost@gmail.com"
print("Project:",PROJECT)
print("Please select this Google account in the login dialog:",EXPECTED_ACCOUNT)
print("Earth Engine Python API:",ee.__version__)
print("A Colab Google Drive mount is NOT needed for this test.")
try:
    ee.Authenticate(auth_mode="colab", force=True)
    print("PASS: Earth Engine authentication completed.")
except Exception as exc:
    print("FAIL: Earth Engine authentication")
    print(type(exc).__name__ + ":",str(exc))
    raise
try:
    ee.Initialize(project=PROJECT)
    print("PASS: ee.Initialize accepted project",PROJECT)
except Exception as exc:
    print("FAIL: ee.Initialize")
    print(type(exc).__name__ + ":",str(exc))
    raise

In [ ]:
# BLOCK 2 — Actual Earth Engine server-side computation
# A successful initialize() alone does not prove the project is operational.
try:
    answer=ee.Number(42).getInfo()
    assert answer==42, f"Unexpected result: {answer}"
    print("PASS: Earth Engine computed 42 using project",PROJECT)
except Exception as exc:
    print("FAIL: Server-side Earth Engine computation")
    print(type(exc).__name__ + ":",str(exc))
    raise


In [ ]:
# BLOCK 3 — Read-only project asset namespace test (no asset writes)
# Asset listing needs additional IAM authorization in some setups.
PARENT=f"projects/{PROJECT}/assets"
try:
    listing=ee.data.listAssets({"parent":PARENT})
    items=listing.get("assets",[]) if isinstance(listing,dict) else []
    print("PASS: Can list assets under",PARENT)
    print("Asset entries found:",len(items))
    for item in items[:20]:
        print(" -",item.get("name"),item.get("type"))
except Exception as exc:
    print("ASSET LIST TEST FAILED:",type(exc).__name__,str(exc))
    print("If BLOCK 2 succeeded, this may reflect asset IAM permissions or")
    print("the project's asset namespace, rather than EE computation access.")


## Interpretation

- **Block 1 fails:** check which account authenticated, Earth Engine registration and Cloud project permissions.
- **Block 1 passes but Block 2 fails:** the project is not yet operational for Earth Engine requests; capture the *exact* error text, which may identify registration, API, or permission problems.
- **Block 2 passes:** `musanga-511110` is usable for Earth Engine computations, regardless of whether the Code Editor displays it in the upper project list.
- **Block 3 fails:** inspect the asset IAM permissions separately; being able to compute does not guarantee permission to list, create or ingest assets.
- **All pass:** you can change the Musanga upload notebook's `CLOUD_PROJECT` and asset paths to `musanga-511110`. Asset creation/upload permissions and GCS staging permissions still need to be tested during upload.

Share the printed PASS/FAIL results and any error text. **Do not share OAuth tokens, credential files, or authentication URLs containing secrets.**

Quick links: [Google Earth Engine Code Editor](https://code.earthengine.google.com/) · [Earth Engine project registration](https://code.earthengine.google.com/register?project=musanga-511110) · [Cloud IAM](https://console.cloud.google.com/iam-admin/iam?project=musanga-511110).
